# 08.1 音频生成的五种思维方式

本 Notebook 用轻量合成音频和架构胶囊图建立本章的总览：逐采样生成、codec token 生成、扩散生成、完整歌曲生成、可控与个性化生成。

本章代码环境的第一步检查。


In [ ]:
from pathlib import Path
import sys

# 路径推断：从 cwd 向上找含 CODE/chapter08/_common 的目录；ROOT 指向 CODE/chapter08/
_p = Path.cwd()
while not (_p / "CODE" / "chapter08" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter08/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
ROOT = _p / "CODE" / "chapter08"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Audio, display

from _common.audio_io import save_audio
from _common.paths import portable_path
from _common.plotting import finish_figure, setup_plot_style
from synthesis.waveforms import waveform_gallery
from visualizations.jasco_conditioning import plot_conditioning_graph
from visualizations.jukebox_hierarchy import plot_jukebox_hierarchy
from visualizations.magnet_masked_generation import plot_masked_generation
from visualizations.riffusion_spectrogram_diffusion import plot_riffusion_process
from visualizations.ssm_scan import plot_scan

OUTPUT_FIGURES = ROOT / "output_figures"
OUTPUT_AUDIO = ROOT / "output_audio" / "08_1"
OUTPUT_FIGURES.mkdir(parents=True, exist_ok=True)
OUTPUT_AUDIO.mkdir(parents=True, exist_ok=True)
setup_plot_style()

def rel(path):
    return portable_path(path, ROOT)


In [ ]:
paradigms = pd.DataFrame(
    [
        {
            "paradigm": "sample autoregression",
            "core_object": "waveform sample",
            "chapter_model": "WaveNet",
            "question": "下一个采样点是什么？",
        },
        {
            "paradigm": "codec-token generation",
            "core_object": "discrete codec token",
            "chapter_model": "EnCodec + MusicGen / mini Codec-LM",
            "question": "压缩后的音频 token 如何续写？",
        },
        {
            "paradigm": "latent diffusion",
            "core_object": "latent noise trajectory",
            "chapter_model": "AudioLDM2 / Stable Audio Open",
            "question": "如何从噪声逐步还原声音？",
        },
        {
            "paradigm": "full-song generation",
            "core_object": "lyrics + long-form structure",
            "chapter_model": "YuE",
            "question": "如何生成有段落结构的完整歌曲？",
        },
        {
            "paradigm": "controlled personalization",
            "core_object": "conditions + adapter weights",
            "chapter_model": "ACE-Step + LoRA",
            "question": "如何把控制条件和个人素材接入生成？",
        },
    ]
)
display(paradigms)


**几类合成波形的最小参照**

正弦波、谐波叠加、扫频信号与噪声脉冲是本章多个 Notebook 共用的已知输入，后面的结构演示与指标检查会反复用到它们。


In [ ]:
sr = 16000
signals = waveform_gallery(duration=1.0, sr=sr)
display_names = {
    "sine": "正弦波",
    "harmonic": "谐波叠加",
    "chirp": "扫频信号",
    "noise_burst": "噪声脉冲",
}

fig, axes = plt.subplots(len(signals), 1, figsize=(9, 1.8 * len(signals)), sharex=True)
for ax, (name, audio) in zip(axes, signals.items()):
    t = np.arange(audio.size) / sr
    ax.plot(t, audio, color="0.15", linewidth=0.8)
    ax.set_ylabel(display_names.get(name, name))
    ax.set_ylim(-1.05, 1.05)
    save_audio(OUTPUT_AUDIO / f"{name}.wav", audio, sr)
axes[-1].set_xlabel("时间（秒）")
finish_figure(fig, OUTPUT_FIGURES / "08_1_generation_paradigms_waveforms.png")
plt.show()

display(Audio(str(OUTPUT_AUDIO / "harmonic.wav")))


**Jukebox 式分层编解码器/先验结构示意**


In [ ]:
fig = plot_jukebox_hierarchy(OUTPUT_FIGURES / "08_1_jukebox_hierarchy.png")
display(fig)
plt.close(fig)


**Riffusion 式频谱图扩散过程示意**


In [ ]:
fig = plot_riffusion_process(OUTPUT_FIGURES / "08_1_riffusion_process.png")
display(fig)
plt.close(fig)


**MAGNeT 式遮蔽 token 生成示意**


In [ ]:
fig = plot_masked_generation(OUTPUT_FIGURES / "08_1_magnet_masked_generation.png")
display(fig)
plt.close(fig)


**JASCO 式多条件控制示意**


In [ ]:
fig = plot_conditioning_graph(OUTPUT_FIGURES / "08_1_jasco_conditioning.png")
display(fig)
plt.close(fig)


**SSM/Mamba 式长序列 scan 机制示意**


In [ ]:
fig = plot_scan(OUTPUT_FIGURES / "08_1_ssm_scan.png")
display(fig)
plt.close(fig)


In [ ]:
generated = sorted(p.name for p in OUTPUT_FIGURES.glob("08_1_*.png"))
print("08_1 generated figures:")
for name in generated:
    print("-", name)
print("08_1 generated audio:")
for path in sorted(OUTPUT_AUDIO.glob("*.wav")):
    print("-", rel(path))
